*0.4 Deep learning basics*

# Transformer: decoder

**The situation.** An encoder lets every token see every other — including the ones that come *after* it. For generating text that is cheating: when predicting the next word, the model must not see it. A decoder is an encoder with one change that makes it a language model.

**The decoder.** Self-attention with a *causal mask*: position *t* may attend to positions 0…*t* and nothing later. Train it to predict the next token at every position at once (all positions in parallel, thanks to the mask). At generation time, feed it a prefix, take the last position's prediction, append, repeat. GPT is exactly this.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The mask, and proof that the future cannot leak.** Change a later token and check that earlier outputs do not move.

In [2]:
import torch
from torch import nn

length = 6
causal_mask = nn.Transformer.generate_square_subsequent_mask(length)
print("causal mask (0 = may attend, -inf = blocked):")
print(causal_mask)

torch.manual_seed(0)
layer = nn.TransformerEncoderLayer(
    d_model=32, nhead=4, dim_feedforward=64, dropout=0.0, batch_first=True
).eval()
tokens = torch.randn(1, length, 32)
changed = tokens.clone()
changed[0, 5] = torch.randn(32)  # change only the LAST token

with torch.no_grad():
    original = layer(tokens, src_mask=causal_mask)
    after_change = layer(changed, src_mask=causal_mask)
    no_mask_original = layer(tokens)
    no_mask_after = layer(changed)

print(
    "with causal mask:    positions 0–4 unchanged →",
    torch.allclose(original[0, :5], after_change[0, :5], atol=1e-6),
)
print(
    "without causal mask: positions 0–4 unchanged →",
    torch.allclose(no_mask_original[0, :5], no_mask_after[0, :5], atol=1e-6),
)
assert torch.allclose(original[0, :5], after_change[0, :5], atol=1e-6) and not torch.allclose(
    no_mask_original[0, :5], no_mask_after[0, :5], atol=1e-6
)

causal mask (0 = may attend, -inf = blocked):
tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])
with causal mask:    positions 0–4 unchanged → True
without causal mask: positions 0–4 unchanged → False


**Reading the output.** The mask is a triangle: each row (a query position) is allowed to see its column and everything to the left. Changing the last token leaves positions 0–4 untouched under the mask, and changes them without it. That guarantee is what makes next-token training honest.

**Next-token training in one line.** Shift the targets by one: the input at position *t* predicts the token at *t+1*.

In [3]:
import torch.nn.functional as F

vocabulary = 100
token_ids = torch.randint(0, vocabulary, (1, length + 1))  # a sequence of 7 token ids
inputs, targets = token_ids[:, :-1], token_ids[:, 1:]  # positions 0..5 predict positions 1..6
print("inputs: ", inputs[0].tolist())
print("targets:", targets[0].tolist(), "(the same sequence, shifted one to the left)")

embedding = nn.Embedding(vocabulary, 32)
head = nn.Linear(32, vocabulary)
logits = head(
    layer(embedding(inputs), src_mask=causal_mask)
)  # (1, 6, vocabulary): a prediction at every position
loss = F.cross_entropy(logits.reshape(-1, vocabulary), targets.reshape(-1))
print(
    "loss for an untrained model:",
    round(loss.item(), 2),
    "≈ ln(100) =",
    round(torch.log(torch.tensor(100.0)).item(), 2),
)
assert abs(loss.item() - 4.6) < 1

inputs:  [74, 6, 1, 73, 5, 63]
targets: [6, 1, 73, 5, 63, 21] (the same sequence, shifted one to the left)
loss for an untrained model: 4.6 ≈ ln(100) = 4.61


```
position   0     1     2     3     4     5
sees      [0]  [0,1] [0-2] [0-3] [0-4] [0-5]      causal: only the past
predicts   t1    t2    t3    t4    t5    t6        one training signal per position
```

**The rule to remember.** Decoder = encoder + causal mask + next-token targets. All positions train in parallel; generation runs one token at a time.

| Use it when | Don't when | Instead use |
|---|---|---|
| generating text, code, anything sequential | reading tasks where seeing both sides helps (classification, retrieval) | encoder |

**Watch out**
- Forget the mask and the model learns to copy the next token from the input; training loss goes to zero and generation is garbage.
- The original transformer decoder also has *cross*-attention to an encoder (translation); GPT-style "decoder-only" models drop it.
- At generation time, the KV cache stores past keys and values so each new token costs one step, not a full re-read.